# TASK 4: CONTENT-BASED FILTERING (OPTIMIZED K=100)

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("TASK 4: CONTENT-BASED FILTERING (OPTIMIZED K=100)")

TASK 4: CONTENT-BASED FILTERING (OPTIMIZED K=100)


## LOAD TF-IDF MATRIX VÀ DATA

In [2]:
print("\nLoad TF-IDF matrix và dữ liệu...")

# Load cleaned movies data
movies = pd.read_csv('../data/cleaned/movies_cleaned.csv')
print(f"Đã load movies: {len(movies):,} bộ phim")

# Load TF-IDF matrix
with open('../models/tfidf_matrix.pkl', 'rb') as f:
    tfidf_matrix = pickle.load(f)
print(f"Đã load TF-IDF matrix: {tfidf_matrix.shape}")

# Load TF-IDF vectorizer
with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
print(f"Đã load TF-IDF vectorizer: {len(tfidf_vectorizer.get_feature_names_out())} features")

# Load movie indices
with open('../models/movie_indices.pkl', 'rb') as f:
    movie_indices = pickle.load(f)
print(f"Đã load movie indices mapping: {len(movie_indices)} phim")

print(f"\nThông tin TF-IDF Matrix:")
print(f"   - Shape: {tfidf_matrix.shape}")
print(f"   - Số phần tử non-zero: {tfidf_matrix.nnz:,}")
print(f"   - Độ thưa: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print(f"   - Memory (ước tính): {tfidf_matrix.data.nbytes / 1024 / 1024:.2f} MB")



Load TF-IDF matrix và dữ liệu...
Đã load movies: 3,416 bộ phim
Đã load TF-IDF matrix: (3416, 1671)
Đã load TF-IDF vectorizer: 1671 features
Đã load movie indices mapping: 3416 phim

Thông tin TF-IDF Matrix:
   - Shape: (3416, 1671)
   - Số phần tử non-zero: 14,487
   - Độ thưa: 99.75%
   - Memory (ước tính): 0.11 MB


## TÍNH COSINE SIMILARITY (OPTIMIZED K=100)

In [3]:
print("TÍNH COSINE SIMILARITY (OPTIMIZED K=100)")

# Tính cosine similarity cho TOÀN BỘ matrix
print("\nĐang tính cosine similarity matrix...")

cosine_sim_full = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Đã tính xong! Shape: {cosine_sim_full.shape}")

# OPTIMIZE: Chỉ giữ top-K similar movies cho mỗi phim
K = 100
print(f"\nĐang optimize: Chỉ giữ top-{K} similar movies cho mỗi phim...")

# Tạo sparse similarity matrix (chỉ lưu top-K)
top_k_indices = []
top_k_scores = []

for i in range(cosine_sim_full.shape[0]):
    # Lấy similarity scores của phim i với tất cả phim khác
    sim_scores = cosine_sim_full[i]
    
    # Lấy top-K indices (không bao gồm chính nó)
    top_indices = np.argsort(sim_scores)[::-1][1:K+1]  # Bỏ index 0 (chính nó)
    top_scores = sim_scores[top_indices]
    
    top_k_indices.append(top_indices)
    top_k_scores.append(top_scores)
    
    if (i + 1) % 500 == 0:
        print(f"Đã xử lý {i+1}/{cosine_sim_full.shape[0]} phim...")

# Convert to numpy arrays
top_k_indices = np.array(top_k_indices)
top_k_scores = np.array(top_k_scores)

print(f"\nOptimization hoàn tất!")
print(f"   - Shape indices: {top_k_indices.shape}")
print(f"   - Shape scores: {top_k_scores.shape}")
print(f"   - Memory tiết kiệm: {(cosine_sim_full.nbytes - (top_k_indices.nbytes + top_k_scores.nbytes)) / 1024 / 1024:.2f} MB")

# Xóa full matrix để giải phóng RAM
del cosine_sim_full
print(f"   - Đã xóa full matrix khỏi RAM")

TÍNH COSINE SIMILARITY (OPTIMIZED K=100)

Đang tính cosine similarity matrix...
Đã tính xong! Shape: (3416, 3416)

Đang optimize: Chỉ giữ top-100 similar movies cho mỗi phim...
Đã xử lý 500/3416 phim...
Đã xử lý 1000/3416 phim...
Đã xử lý 1500/3416 phim...
Đã xử lý 2000/3416 phim...
Đã xử lý 2500/3416 phim...
Đã xử lý 3000/3416 phim...

Optimization hoàn tất!
   - Shape indices: (3416, 100)
   - Shape scores: (3416, 100)
   - Memory tiết kiệm: 83.82 MB
   - Đã xóa full matrix khỏi RAM


## BUILD RECOMMENDATION FUNCTION

In [4]:
print("XÂY DỰNG HÀM GỢI Ý")

def content_based_recommend(movie_id=None, movie_title=None, n=10, verbose=True):
    """
    Gợi ý phim dựa trên nội dung (Content-Based Filtering)
    
    Parameters:
    -----------
    movie_id : int
        ID của phim cần tìm phim tương tự
    movie_title : str
        Tên phim (alternative to movie_id)
    n : int
        Số lượng phim gợi ý (default=10)
    verbose : bool
        Hiển thị thông tin chi tiết
    
    Returns:
    --------
    DataFrame: Danh sách phim gợi ý
    """
    
    # Tìm movie index
    if movie_title:
        # Tìm theo title
        matches = movies[movies['title_clean'].str.contains(movie_title, case=False, na=False)]
        if len(matches) == 0:
            print(f"Không tìm thấy phim: '{movie_title}'")
            return None
        movie_id = matches.iloc[0]['movieId']
        if verbose:
            print(f"Tìm thấy: {matches.iloc[0]['title_clean']}")
    
    # Kiểm tra movie_id có trong mapping không
    if movie_id not in movie_indices:
        print(f"Movie ID {movie_id} không tồn tại trong dataset")
        return None
    
    # Lấy index của movie trong matrix
    idx = movie_indices[movie_id]
    
    # Lấy thông tin phim gốc
    movie_info = movies.iloc[idx]
    
    if verbose:
        print(f"\nPhim gốc:")
        print(f"   - Tên: {movie_info['title_clean']}")
        print(f"   - Thể loại: {movie_info['genres']}")
        print(f"   - Rating: {movie_info['rating_avg']:.2f}/5.0")
        print(f"   - Số lượt rate: {movie_info['rating_count']:.0f}")
    
    # Lấy top-K similar movies đã tính sẵn
    similar_indices = top_k_indices[idx]
    similar_scores = top_k_scores[idx]
    
    # Lấy top N
    top_n_indices = similar_indices[:n]
    top_n_scores = similar_scores[:n]
    
    # Tạo DataFrame kết quả
    recommendations = movies.iloc[top_n_indices].copy()
    recommendations['similarity_score'] = top_n_scores
    recommendations['rank'] = range(1, n+1)
    
    # Sắp xếp lại columns
    result_cols = ['rank', 'movieId', 'title_clean', 'genres', 'rating_avg', 
                   'rating_count', 'similarity_score']
    recommendations = recommendations[result_cols]
    
    if verbose:
        print(f"\nTop {n} phim tương tự:")
        print("-" * 100)
        for _, row in recommendations.iterrows():
            print(f"#{row['rank']:<2} | {row['title_clean']:<40} | {row['genres']:<30} | "
                  f"Rating: {row['rating_avg']:.2f} | Sim: {row['similarity_score']:.3f}")
    
    return recommendations


def content_based_recommend_multi(movie_ids, n=10, verbose=True):
    """
    Gợi ý phim dựa trên nhiều phim yêu thích (Cold Start)
    
    Parameters:
    -----------
    movie_ids : list
        List các movie_id yêu thích
    n : int
        Số lượng phim gợi ý
    verbose : bool
        Hiển thị thông tin chi tiết
    
    Returns:
    --------
    DataFrame: Danh sách phim gợi ý
    """
    
    if verbose:
        print(f"\nGợi ý dựa trên {len(movie_ids)} phim yêu thích:")
    
    # Tính average similarity
    all_scores = np.zeros(len(movies))
    valid_count = 0
    
    for movie_id in movie_ids:
        if movie_id not in movie_indices:
            if verbose:
                print(f"Movie ID {movie_id} không tồn tại, bỏ qua...")
            continue
        
        idx = movie_indices[movie_id]
        movie_title = movies.iloc[idx]['title_clean']
        
        if verbose:
            print(f"{movie_title}")
        
        # Lấy similarity scores
        similar_indices = top_k_indices[idx]
        similar_scores = top_k_scores[idx]
        
        # Cộng scores vào all_scores
        all_scores[similar_indices] += similar_scores
        valid_count += 1
    
    if valid_count == 0:
        print("Không có phim hợp lệ nào!")
        return None
    
    # Tính average
    all_scores /= valid_count
    
    # Loại bỏ các phim đã có trong input
    input_indices = [movie_indices[mid] for mid in movie_ids if mid in movie_indices]
    all_scores[input_indices] = -1
    
    # Lấy top N
    top_n_indices = np.argsort(all_scores)[::-1][:n]
    top_n_scores = all_scores[top_n_indices]
    
    # Tạo DataFrame kết quả
    recommendations = movies.iloc[top_n_indices].copy()
    recommendations['avg_similarity'] = top_n_scores
    recommendations['rank'] = range(1, n+1)
    
    result_cols = ['rank', 'movieId', 'title_clean', 'genres', 'rating_avg', 
                   'rating_count', 'avg_similarity']
    recommendations = recommendations[result_cols]
    
    if verbose:
        print(f"\nTop {n} phim gợi ý:")
        print("-" * 100)
        for _, row in recommendations.iterrows():
            print(f"#{row['rank']:<2} | {row['title_clean']:<40} | {row['genres']:<30} | "
                  f"Rating: {row['rating_avg']:.2f} | Sim: {row['avg_similarity']:.3f}")
    
    return recommendations


print("Đã tạo 2 functions:")
print("   1. content_based_recommend(movie_id, n=10)")
print("   2. content_based_recommend_multi(movie_ids, n=10)")

XÂY DỰNG HÀM GỢI Ý
Đã tạo 2 functions:
   1. content_based_recommend(movie_id, n=10)
   2. content_based_recommend_multi(movie_ids, n=10)


## TEST RECOMMENDATION FUNCTIONS

In [5]:
print("KIỂM TRA HÀM GỢI Ý")

# TEST 1: Gợi ý từ 1 phim
print("TEST 1: GỢI Ý TỪ 1 PHIM")

# Lấy một phim phổ biến để test
test_movie = movies[movies['rating_count'] > 1000].sample(1).iloc[0]
test_movie_id = test_movie['movieId']

print(f"\nTest với phim: {test_movie['title_clean']}")
print(f"   Genre: {test_movie['genres']}")

recs_1 = content_based_recommend(movie_id=test_movie_id, n=10, verbose=True)

# Phân tích kết quả
if recs_1 is not None:
    original_genres = set(test_movie['genres'].split('|'))
    matching_genres = 0
    
    for _, rec in recs_1.iterrows():
        rec_genres = set(rec['genres'].split('|'))
        if original_genres & rec_genres:  # Có genre chung
            matching_genres += 1
    
    print(f"\nPhân tích kết quả:")
    print(f"   - {matching_genres}/{len(recs_1)} phim có genre trùng khớp ({matching_genres/len(recs_1)*100:.1f}%)")
    print(f"   - Similarity score trung bình: {recs_1['similarity_score'].mean():.3f}")

# TEST 2: Gợi ý từ nhiều phim (Cold Start)
print("TEST 2: GỢI Ý TỪ NHIỀU PHIM (COLD START)")

# Lấy 3 phim ngẫu nhiên từ các genre khác nhau
test_movies_action = movies[movies['genres'].str.contains('Action', na=False)].sample(1)
test_movies_comedy = movies[movies['genres'].str.contains('Comedy', na=False)].sample(1)
test_movies_drama = movies[movies['genres'].str.contains('Drama', na=False)].sample(1)

test_movie_ids = [
    test_movies_action.iloc[0]['movieId'],
    test_movies_comedy.iloc[0]['movieId'],
    test_movies_drama.iloc[0]['movieId']
]

recs_2 = content_based_recommend_multi(test_movie_ids, n=10, verbose=True)

# TEST 3: Test bằng tên phim
print("TEST 3: TÌM PHIM BẰNG TÊN")

# Test với một phim nổi tiếng
recs_3 = content_based_recommend(movie_title="Toy Story", n=5, verbose=True)

KIỂM TRA HÀM GỢI Ý
TEST 1: GỢI Ý TỪ 1 PHIM

Test với phim: Gone with the Wind
   Genre: Drama|Romance|War

Phim gốc:
   - Tên: Gone with the Wind
   - Thể loại: Drama|Romance|War
   - Rating: 4.00/5.0
   - Số lượt rate: 1156

Top 10 phim tương tự:
----------------------------------------------------------------------------------------------------
#1  | Rob Roy                                  | Drama|Romance|War              | Rating: 3.59 | Sim: 0.633
#2  | Casablanca                               | Drama|Romance|War              | Rating: 4.41 | Sim: 0.633
#3  | Inherit the Wind                         | Drama                          | Rating: 4.28 | Sim: 0.573
#4  | Lucie Aubrac                             | Romance|War                    | Rating: 3.67 | Sim: 0.556
#5  | For the Moment                           | Romance|War                    | Rating: 3.40 | Sim: 0.556
#6  | Colonel Chabert, Le                      | Drama|Romance|War              | Rating: 3.70 | Sim: 0.524
#7 

## LƯU MODEL

In [6]:
print("LƯU CONTENT-BASED MODEL")

# Tạo model object
content_based_model = {
    'top_k_indices': top_k_indices,
    'top_k_scores': top_k_scores,
    'movie_indices': movie_indices,
    'K': K,
    'n_movies': len(movies),
    'tfidf_features': len(tfidf_vectorizer.get_feature_names_out())
}

# Save model
with open('../models/content_based_model.pkl', 'wb') as f:
    pickle.dump(content_based_model, f)

print("Đã lưu: models/content_based_model.pkl")
print(f"   - Top-{K} indices: {top_k_indices.shape}")
print(f"   - Top-{K} scores: {top_k_scores.shape}")
print(f"   - File size: {(top_k_indices.nbytes + top_k_scores.nbytes) / 1024 / 1024:.2f} MB")




LƯU CONTENT-BASED MODEL
Đã lưu: models/content_based_model.pkl
   - Top-100 indices: (3416, 100)
   - Top-100 scores: (3416, 100)
   - File size: 5.21 MB


## TỔNG KẾT

In [7]:
print("TASK 4 HOÀN THÀNH!")

print("\nChecklist:")
print("Load TF-IDF matrix từ Task 2")
print("Tính Cosine Similarity (optimized K=100)")
print("Build recommendation functions")
print("Test với nhiều trường hợp")
print("Lưu model và functions")

print("\nOutput files:")
print("  - models/content_based_model.pkl")

print("\nFunctions available:")
print("  - content_based_recommend(movie_id, n=10)")
print("  - content_based_recommend_multi(movie_ids, n=10)")

print("\nModel Stats:")
print(f"Số phim: {len(movies):,}")
print(f"Top-K per movie: {K}")
print(f"TF-IDF features: {len(tfidf_vectorizer.get_feature_names_out())}")
print(f"Model size: {(top_k_indices.nbytes + top_k_scores.nbytes) / 1024 / 1024:.2f} MB")



TASK 4 HOÀN THÀNH!

Checklist:
Load TF-IDF matrix từ Task 2
Tính Cosine Similarity (optimized K=100)
Build recommendation functions
Test với nhiều trường hợp
Lưu model và functions

Output files:
  - models/content_based_model.pkl

Functions available:
  - content_based_recommend(movie_id, n=10)
  - content_based_recommend_multi(movie_ids, n=10)

Model Stats:
Số phim: 3,416
Top-K per movie: 100
TF-IDF features: 1671
Model size: 5.21 MB
